In [1]:
import torch
import torch.nn as nn
import math

In [2]:
class InputEmbeddig(nn.Module):

  def __init__(self,d_model: int,vocab_size:int):
    super().__init__()
    self.d_model = d_model #size 512
    self.vocab_size = vocab_size #input id
    self.embedding = nn.embedding(vocab_size,d_model)

  def forward(self,x):
    return self.embedding(x) * math.sqrt(self.d_model)

In [3]:
class positionalEncoding(nn.Module):

  def _init_(self,seq_len,d_model,dropout):
    super.__init__()
    self.seq_len = seq_len
    self.d_model = d_model
    self.dropout = dropout

    #created a metric of shape (seq_len, d_model)(6,512)
    pe = torch.zeros(seq_len, d_model)

    #create a vector of length [1,seq_len] why this length or dim
    #position / div_term → shape: [4, 1] / [1, d_model] → broadcasts to [4, d_model]

    position = torch.arange(0,seq_len,dtype=torch.float).unsqueeze(1)

    #create a dinominator for position output will be (d_model,2)

    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

    # Apply sine to even indices
    pe[:,0::2] = torch.sin(position/div-term)# sin(position * (10000 ** (2i / d_model))
    # Apply cosine to odd indices
    pe[:, 1::2] = torch.cos(position * div_term) # cos(position * (10000 ** (2i / d_model))

    pe = pe.unsqueeze(0) #(1,seq_len,d_model)

    #register it in a buffer becuase u don't want to recreate it. Once created and saved using buffer
    #it can be used for all the postions
    #Always remember position encoding is for indexes not for words

    self.register_buffer('pe',pe)

    def forward(self,x):
      x = x + (self.pe[:,:x.shape[1],:]).requires_grad_(False) #requires_grad is for we are telling the model that it don't want to learn this postion encoding
      return self.dropout(x)



In [4]:
class Layernormalization(nn.Module): #i'm not getting the whole block
  def __init__(self,eps= 10**-6):
    super().__init__()
    self.eps = eps
    self.alpha = nn.Parameter(torch.ones(1)) #alpha is a Multipliable
    self.bias = nn.Parameter(torch.zeros(0)) #bias is a additive

  def forward(self,x):
    mean = x.mean(dim = -1 , keepdim=True)
    std = x.std(dim = -1 , keepdim=True)
    return self.alpha * (x - mean)/(std + self.eps) + self.bias

In [5]:
class FeedForwardBlock(nn.Module):  #i'm not getting the whole block

    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff) # w1 and b1
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model) # w2 and b2

    def forward(self, x):
        # (batch, seq_len, d_model) --> (batch, seq_len, d_ff) --> (batch, seq_len, d_model)
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

In [6]:
class Multiheadedattention(nn.Module):
  def __init__(self, d_model, h,dropout) -> None:
    super().__init__()
    self.d_model = d_model
    self.h = h
    assert d_model % h == 0, "d_model can't be divided into even parts"


    self.dk = d_model//h
    self.w_q = nn.Linear(d_model,d_model)
    self.w_k = nn.Linear(d_model,d_model)
    self.w_v = nn.Linear(d_model,d_model)

    self.w_o = nn.Linear(d_model,d_model)
    self.dropout = nn.Dropout(dropout)

  @staticmethod
  def attention(self,query,key,value,mask,dropout):
    dk = query.shape[-1] #(batch,h,seq_len,dk) we get the dk from last so
    attention_scores = (query @ key.transpose(-2,-1)) // math.sqrt(dk) #(batch,h,dk,seq_len)
    if mask is not None:
      attention_scores.masked_fill_(mask == 0, -1e9) #i'm not getting maskedfill(mask == 0)
    attention_scores = attention_scores.softmax(dim = -1) #(batch,h,seq_len,seq_len)
    if dropout is not None:
      attention_scores  = dropout(attention_scores)

    return (attention_scores @ value), attention_scores


  def forward(self,q,k,v,mask):
    query = self.w_q(q) #(Batchlen, seq_len, d_model) -> (Batchlen, seq_len, d_model) why? becuase we need to rearange the 512 embedding vector to new from for q,k,v
    key = self.w_k(k) #same as above
    value = self.w_v(v) #same as above

    #we need to divid the q,k,v into small vectors so that we can give it to head

    #(batch,seq_len,d_model) ---> (batch,seq_len,h,dk) where h*dk = d_model --->with the help of transpose (batch,h,seq_len,dk) why?
    # becuase imagine 1st one box as batch len inside there will be another dimension h which is head inside it there will be a vector whose dimenstion are (6,128)
    query = query.view(query.shape[0],query.shape[1],self.h,self.dk).transpose(1,2)
    key = key.view(query.shape[0],query.shape[1],self.h,self.dk).transpose(1,2)
    value = value.view(query.shape[0],query.shape[1],self.h,self.dk).transpose(1,2)

    x,self.attention_scores = Multiheadedattention.attention(query,key,value,mask,self.dropout)


In [7]:
class ResuidualConnection(nn.Module):

  def __init__(self, dropout) -> None:
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    self.norm = Layernormalization()

  def forward(self,x,sublayer):
    return x + self.dropout(sublayer(self.norm)) #i'm not getting the sublayer(self.norm)

In [8]:
class EncoderBlock(nn.Module):

  def __init__(self, self_attention_block: Multiheadedattention ,feed_forward_block: FeedForwardBlock,dropout) -> None:
    super().__init__()
    self.self_attention_block = self_attention_block
    self.feed_forward_block = FeedForwardBlock
    self.ResuidualConnection = nn.ModuleList([ResuidualConnection(dropout) for _ in range(2)])

  def forward(self,x,src_mask):
    x = self.ResuidualConnection[0](x, lambda x:self.self_attention_block(x,x,x,src_mask))
    x = self.ResuidualConnection[1](x,self.feed_forward_block)
    return x

In [9]:
class Encoder(nn.Module):

  def __init__(self, layers:nn.ModuleList) -> None:
    super().__init__()
    self.layer = layers
    self.norm = Layernormalization()

  def forwar(self,x,mask):
    for layer in self.layer:
      x = layer(x,mask)
    return self.norm(x)

In [10]:
class DecoderBlock(nn.Module):

  def __init__(self, self_attention_block:Multiheadedattention,Cross_attention_block:Multiheadedattention,feed_forward_block:FeedForwardBlock,dropout) -> None:
    super().__init__()
    self.self_attention_block = self_attention_block
    self.Cross_attention_block = Cross_attention_block
    self.feed_forward_block = feed_forward_block
    self.ResuidualConnection = nn.ModuleList([ResuidualConnection(dropout) for _ in range(3)])

  def forward(self,x,encoder_ouput,src_mask,tgt_mask):
    x = self.ResuidualConnection[0](x, lambda x:self.self_attention_block(x,x,x,tgt_mask))
    x = self.ResuidualConnection[1](x, lambda x:self.self_attention_block(x,encoder_ouput,encoder_ouput,src_mask))
    x = self.ResuidualConnection[2](x, self.feed_forward_block)
    return x

In [11]:
class Decoder(nn.Module):

  def __init__(self, layers:nn.ModuleList) -> None:
    super().__init__()
    self.layer = layers
    self.norm = Layernormalization()

  def forward(self, x, encoder_ouput,src_mask,tgt_mask):
    for layer in self.layer:
      x = layer(x,encoder_ouput,src_mask,tgt_mask)
    return self.norm(x)



In [12]:
class projectionLayer(nn.Module):

  def __init__(self, d_model,vocab_size) -> None:
    super().__init__()
    self.proj = nn.Linear(d_model,vocab_size)

  def forward(self,x):
    return torch.log_softmax(self.proj(x),dim = -1)

In [19]:
class Transformer(nn.Module):

    def __init__(self, encoder: Encoder, decoder: Decoder, src_embed: InputEmbeddig, tgt_embed: InputEmbeddig, src_pos: positionalEncoding, tgt_pos: positionalEncoding, projection_layer: projectionLayer) -> None:
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.projection_layer = projection_layer

    def encode(self, src, src_mask):
        # (batch, seq_len, d_model)
        src = self.src_embed(src)
        src = self.src_pos(src)
        return self.encoder(src, src_mask)

    def decode(self, encoder_output: torch.Tensor, src_mask: torch.Tensor, tgt: torch.Tensor, tgt_mask: torch.Tensor):
        # (batch, seq_len, d_model)
        tgt = self.tgt_embed(tgt)
        tgt = self.tgt_pos(tgt)
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)

    def project(self, x):
        # (batch, seq_len, vocab_size)
        return self.projection_layer(x)


In [20]:
def build_transformer(src_vocab_size: int, tgt_vocab_size: int, src_seq_len: int, tgt_seq_len: int, d_model: int=512, N: int=6, h: int=8, dropout: float=0.1, d_ff: int=2048) -> Transformer:
        # Create the embedding layers
        src_embed = InputEmbeddings(d_model, src_vocab_size)
        tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)

        # Create the positional encoding layers
        src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
        tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout)

        # Create the encoder blocks
        encoder_blocks = []
        for _ in range(N):
            encoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
            feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
            encoder_block = EncoderBlock(d_model, encoder_self_attention_block, feed_forward_block, dropout)
            encoder_blocks.append(encoder_block)

        # Create the decoder blocks
        decoder_blocks = []
        for _ in range(N):
            decoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
            decoder_cross_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
            feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
            decoder_block = DecoderBlock(d_model, decoder_self_attention_block, decoder_cross_attention_block, feed_forward_block, dropout)
            decoder_blocks.append(decoder_block)

        # Create the encoder and decoder
        encoder = Encoder(d_model, nn.ModuleList(encoder_blocks))
        decoder = Decoder(d_model, nn.ModuleList(decoder_blocks))

        # Create the projection layer
        projection_layer = ProjectionLayer(d_model, tgt_vocab_size)

        # Create the transformer
        transformer = Transformer(encoder, decoder, src_embed, tgt_embed, src_pos, tgt_pos, projection_layer)

        # Initialize the parameters
        for p in transformer.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

        return transformer